## Setup (installs, imports, config)

In [3]:
# --- Install deps (first run may take a few minutes) ---
!pip install unsloth
!pip -q install "transformers>=4.56.0" datasets soundfile librosa accelerate peft contractions --upgrade

# --- Install missing package ---
%pip install librosa

# --- Imports & config ---
import os, re, unicodedata, numpy as np, torch, soundfile as sf, librosa
from datasets import load_dataset, Audio as DS_Audio
from huggingface_hub import login
from unsloth import FastModel, is_bfloat16_supported
from transformers import CsmForConditionalGeneration, AutoProcessor, TrainingArguments, Trainer
import contractions as ct
from IPython.display import Audio, display

# (Optional) If you must access a gated repo, paste your HF token; otherwise leave ""
HF_TOKEN = ""

# Training knobs (tune as you like)
DATASET_SIZE = 200         # how many LJSpeech samples to use for quick fine-tune (increase for quality)
MAX_STEPS    = 300         # training steps (increase for quality; eg 1k–3k if you have time/GPU)
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-4
WARMUP       = 10
SEED         = 3407
OUT_DIR      = "csm_lora_out"

# Long-form synthesis knobs
TARGET_SECONDS = 300       # ≈ 5 minutes
STEP_SECONDS   = 12        # per continuation step (~10–15 s typical cap)
CROSSFADE_MS   = 80        # smooth join
TAIL_MS        = 450       # tiny tail fed back as style/context

# Your 5-min script (paste your content here)
LONG_TEXT = (
"Thanks for calling. I will walk you through a complete step-by-step guide so everything is clear and easy to follow. "
"First, let us confirm your account email and make sure it is the same one you used during registration. If you are unsure, open Profile and verify the primary email on file. "
"Next, if you are getting a sign-in error, use the Forgot password link and watch for the reset email. If nothing arrives within a minute, check spam or promotions, then try again. "
"Now, notifications. Open Settings and ensure app alerts are enabled. If you still do not see updates, toggle them off and back on, then relaunch the app. "
"Check Background App Refresh or battery optimization—on some devices, strict power settings delay notifications. If needed, add the app to the allowed list. "
"Let us refresh your data. Go to the Dashboard or Activity tab and pull to refresh so the latest information syncs. If something looks stale, try a manual refresh and verify your connection. "
"For Wi-Fi, toggle it off and on; for cellular, confirm data is enabled and the signal is strong. If performance feels sluggish, clear temporary cache from app settings and restart the app. "
"If the app is behind on updates, install the latest version—updates often improve stability, fix minor bugs, and enhance voice features. "
"Here is a quick checklist: email confirmed, password reset if needed, notifications enabled, battery optimization adjusted, network checked, cache cleared, and app updated. "
"If a single step does not help, combine two or three—like updating the app and restarting the device. "
"If you are using voice features, record a short test in a quiet room. Listen for clarity, pacing, and emphasis, and add short pauses between sentences for a calmer tone. "
"If you prefer a more energetic sound, keep sentences concise and direct. "
"Finally, if anything still does not behave as expected, send a brief description and a short screen recording. I will review it and follow up with next steps so we can wrap this up quickly and get you back on track."
)

# Login if token provided (safe to skip if blank)
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print("HF login warning:", e)

device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", device)


  Using cached unsloth-2025.9.11-py3-none-any.whl.metadata (55 kB)
  Using cached unsloth_zoo-2025.9.14-py3-none-any.whl.metadata (31 kB)
  Using cached xformers-0.0.32.post2.tar.gz (12.1 MB)
  Using cached unsloth_zoo-2025.9.14-py3-none-any.whl.metadata (31 kB)
  Using cached xformers-0.0.32.post2.tar.gz (12.1 MB)
  Installing build dependencies ...   Installing build dependencies ... -done
  Getting requirements to build wheel ... one
  Getting requirements to build wheel ... -done
  Preparing metadata (pyproject.toml) ... one
  Preparing metadata (pyproject.toml) ... -done
done
  Using cached tyro-0.9.32-py3-none-any.whl.metadata (11 kB)
  Using cached tyro-0.9.32-py3-none-any.whl.metadata (11 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached trl-0.23.0-py3-none-any.whl.metadata (11 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached trl-0.23.0-py3-none-any.whl.metadata (11 k

ModuleNotFoundError: No module named 'unsloth'

## Download LJSpeech (English) and prepare audio

In [ ]:
# LJSpeech: English single-speaker corpus. We’ll resample to 24 kHz and add a speaker id ("0")
ds_raw = load_dataset("keithito/lj_speech", split="train")
ds_raw = ds_raw.add_column("source", ["0"] * len(ds_raw))
ds_raw = ds_raw.cast_column("audio", DS_Audio(sampling_rate=24_000))
if DATASET_SIZE > 0 and DATASET_SIZE < len(ds_raw):
    ds_raw = ds_raw.shuffle(seed=SEED).select(range(DATASET_SIZE))

print(ds_raw)
print("Example text:", ds_raw[0]["text"][:120])


## Load CSM-1B via Unsloth, add LoRA, preprocess, and train

In [ ]:
# Load base model & processor (Unsloth mirror of CSM-1B)
model, processor = FastModel.from_pretrained(
    model_name     = "unsloth/csm-1b",
    max_seq_length = 2048,
    dtype          = None,
    auto_model     = CsmForConditionalGeneration,
    load_in_4bit   = False,
)

# Attach LoRA adapters
model = FastModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.to(device).train()

# Preprocess into the model’s chat format (audio + text per example)
def _preprocess(ex):
    convo = [{
        "role": str(ex["source"]),
        "content": [
            {"type": "text", "text": ex["text"]},
            {"type": "audio","path": ex["audio"]["array"]},
        ],
    }]
    out = processor.apply_chat_template(
        convo, tokenize=True, return_dict=True, output_labels=True,
        text_kwargs   = dict(padding="max_length", max_length=256, pad_to_multiple_of=8, padding_side="right"),
        audio_kwargs  = dict(sampling_rate=24_000, max_length=240001, padding="max_length"),
        common_kwargs = dict(return_tensors="pt"),
    )
    keys = ["input_ids","attention_mask","labels","input_values","input_values_cutoffs"]
    return {k: out[k][0] for k in keys}

ds_train = ds_raw.map(_preprocess, remove_columns=ds_raw.column_names, desc="Preparing dataset")

# Train LoRA
args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP,
    max_steps=MAX_STEPS,
    learning_rate=LR,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    output_dir=OUT_DIR,
    report_to="none",
    save_strategy="no",
)
trainer = Trainer(model=model, train_dataset=ds_train, args=args)
train_result = trainer.train()
print("Training done. Runtime (s):", train_result.metrics.get("train_runtime"))

# Save adapters & processor (reuse later without retraining)
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
model.eval().to(device)


## Long-form inference (~5 min) with seamless continuation

In [ ]:
# Small helpers (inline, no external functions)
def _normalize_text(t: str) -> str:
    t = unicodedata.normalize("NFKC", t)
    t = ct.fix(t)                      # expand contractions safely (you’re -> you are)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def _crossfade(a, b, ms, rate=24_000):
    if len(a) == 0: return b
    if len(b) == 0: return a
    n = max(1, int(rate * ms / 1000))
    n = min(n, len(a), len(b))
    w = np.linspace(0, 1, n, dtype=np.float32)
    return np.concatenate([a[:-n], a[-n:] * (1 - w) + b[:n] * w, b[n:]])

# Reference audio for style: first item’s audio (you can replace with your own WAV if you want)
ref_audio = ds_raw[0]["audio"]["array"].astype(np.float32)
script    = _normalize_text(LONG_TEXT)
sr        = 24_000

# Split script into ~STEP_SECONDS chunks based on ~165 wpm
words = script.split()
words_per_step = max(20, int(165 * (STEP_SECONDS / 60)))
segments, buf = [], []
for w in words:
    buf.append(w)
    if len(buf) >= words_per_step:
        segments.append(" ".join(buf)); buf = []
if buf: segments.append(" ".join(buf))

# Generate sequentially; feed a tiny tail as the next reference; crossfade seams
final_audio = np.zeros(0, dtype=np.float32)
for i, seg in enumerate(segments, 1):
    ref_tail = None
    if final_audio.size > 0:
        n_tail = int(sr * TAIL_MS / 1000)
        ref_tail = final_audio[-n_tail:].copy()

    conversation = [
        {"role": "0", "content": [{"type":"text","text":" "},
                                  {"type":"audio","path": ref_tail if ref_tail is not None else ref_audio}]},
        {"role": "0", "content": [{"type":"text","text": seg}]},
    ]
    inputs = processor.apply_chat_template(conversation, tokenize=True, return_dict=True)
    inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            output_audio=True,
            do_sample=True, temperature=0.9, top_p=0.95,
            max_new_tokens=20000, max_length=20000,
        )

    # get waveform
    try:
        wav = out[0].to(torch.float32).cpu().numpy()
    except Exception:
        tmp = "_tmp.wav"
        processor.save_audio(out, tmp)
        wav, _ = librosa.load(tmp, sr=sr, mono=True)
        os.remove(tmp)

    final_audio = _crossfade(final_audio, wav, CROSSFADE_MS, sr)
    dur = final_audio.size / sr
    print(f"Step {i}: seg ~{STEP_SECONDS}s | total ~{dur:.1f}s")
    if dur >= TARGET_SECONDS:
        break

# Save & play
out_path = "finetuned_5min.wav"
sf.write(out_path, final_audio, sr, subtype="PCM_16")
display(Audio(out_path, rate=sr))
print("Saved:", out_path, "| duration ~", round(final_audio.size / sr, 1), "s")
